# ⚡ Data Pipeline Anomaly Detection (Kafka/Spark Logs)
**Author:** Shiwanshu Inamdar | Data Engineer

This notebook analyzes the metadata output from a simulated Kafka-to-Spark streaming pipeline. We use Pandas to detect pipeline bottlenecks, late-arriving data, and schema drifts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Simulating pipeline metrics dataset
np.random.seed(42)
times = pd.date_range(start='2023-10-01', periods=100, freq='T')
throughput = np.random.normal(5000, 500, 100)
latency = np.random.normal(45, 10, 100)

# Inject an anomaly (Pipeline bottleneck)
throughput[70:80] = np.random.normal(1000, 200, 10)
latency[70:80] = np.random.normal(250, 50, 10)

df = pd.DataFrame({'timestamp': times, 'throughput_msg_sec': throughput, 'processing_latency_ms': latency})
df.head()

### Visualizing Pipeline Health
Plotting Throughput vs Latency to identify backpressure in the Spark executors.

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))

color = 'tab:blue'
ax1.set_xlabel('Time')
ax1.set_ylabel('Throughput (msg/sec)', color=color)
ax1.plot(df['timestamp'], df['throughput_msg_sec'], color=color, label='Throughput')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  
color = 'tab:red'
ax2.set_ylabel('Latency (ms)', color=color)
ax2.plot(df['timestamp'], df['processing_latency_ms'], color=color, linestyle='--', label='Latency')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Kafka/Spark Pipeline Telemetry - Anomaly Detected')
fig.tight_layout()  
plt.show()

### Conclusion
The visualization clearly shows a bottleneck between minute 70 and 80, where throughput crashed and latency spiked to >250ms, triggering our backpressure alerts.